In [11]:
import duckdb
import pandas as pd

# -----------------------------
# CONNECT TO DB
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# AIRPORT METRICS
# -----------------------------
airport_metrics = conn.sql("""
SELECT

    a.iata_code,
    a.name AS airport_name,
    a.iso_country,
    a.municipality,

    w.temperature_c,
    w.wind_kph,
    w.visibility_km,
    w.weather_condition,

    COUNT(f.icao24)
        AS nearby_flights

FROM airports a

LEFT JOIN weather w
ON TRIM(a.iata_code)
=
TRIM(w.airport_code)

LEFT JOIN flights f
ON ABS(
    a.latitude_deg
    - f.latitude
) < 1

AND ABS(
    a.longitude_deg
    - f.longitude
) < 1

WHERE a.iata_code
IS NOT NULL

GROUP BY ALL
""").df()

# -----------------------------
# CREATE STRESS SCORE
# -----------------------------
airport_metrics[
    "weather_risk"
] = 0

airport_metrics.loc[
    airport_metrics[
        "wind_kph"
    ] > 40,
    "weather_risk"
] += 30

airport_metrics.loc[
    airport_metrics[
        "visibility_km"
    ] < 5,
    "weather_risk"
] += 30

airport_metrics[
    "airport_stress_score"
] = (
    airport_metrics[
        "nearby_flights"
    ] * 0.6
    +
    airport_metrics[
        "weather_risk"
    ] * 0.4
)

# -----------------------------
# SAVE TABLE
# -----------------------------
conn.register(
    "metrics_df",
    airport_metrics
)

conn.execute("""
CREATE OR REPLACE TABLE
airport_metrics AS
SELECT *
FROM metrics_df
""")

print(
    "Airport metrics created!"
)

airport_metrics.head()

Airport metrics created!


,iata_code,airport_name,iso_country,municipality,temperature_c,wind_kph,visibility_km,weather_condition,nearby_flights,weather_risk,airport_stress_score
0,RKV,Reykjavík Domestic Airport,IS,Reykjavík,11.0,19.1,10.0,Sunny,6,0,3.6
1,PRN,Priština Adem Jashari International Airport,XK,Prishtina,14.1,10.4,10.0,Patchy rain nearby,12,0,7.2
2,YAG,Fort Frances Municipal Airport,CA,Fort Frances,12.0,4.0,10.0,Sunny,3,0,1.8
3,YAZ,Tofino / Long Beach Airport,CA,Tofino,11.7,7.6,10.0,Partly Cloudy,4,0,2.4
4,QBC,Bella Coola Airport,CA,Bella Coola,8.2,3.6,10.0,Sunny,2,0,1.2


In [12]:
conn.sql("""
SELECT *
FROM airport_metrics
ORDER BY
airport_stress_score DESC
LIMIT 200
""").df()

,iata_code,airport_name,iso_country,municipality,temperature_c,wind_kph,visibility_km,weather_condition,nearby_flights,weather_risk,airport_stress_score
0,PHX,Phoenix Sky Harbor International Airport,US,Phoenix,28.2,4.7,10.0,Sunny,249,0,149.4
1,LUF,Luke Air Force Base,US,Glendale,28.1,8.6,10.0,Sunny,244,0,146.4
2,MMU,Morristown Municipal Airport,US,Morristown,31.8,19.8,10.0,Sunny,236,0,141.6
3,WRI,Mc Guire Air Force Base,US,Wrightstown,32.7,22.7,10.0,Sunny,235,0,141.0
4,TEB,Teterboro Airport,US,Teterboro,33.0,21.6,10.0,Sunny,235,0,141.0
...,...,...,...,...,...,...,...,...,...,...,...
195,SSI,St Simons Island Airport,US,St Simons Island,27.6,15.5,10.0,Patchy rain nearby,78,0,46.8
196,AUS,Austin Bergstrom International Airport,US,Austin,27.0,12.2,10.0,Light rain shower,78,0,46.8
197,BAB,Beale Air Force Base,US,Beale Air Force Base,23.9,4.3,10.0,Sunny,78,0,46.8
198,LWM,Lawrence Municipal Airport,US,Lawrence,29.7,24.8,10.0,Partly Cloudy,78,0,46.8


In [13]:
conn.close()